# 人脸性别识别


在当今数字化时代，人脸识别技术的应用日益广泛，从安全认证到个性化服务，其重要性不言而喻。本次实训项目聚焦于人脸性别识别技术，旨在通过深度学习方法实现对人脸性别的高效、准确识别。            
人脸性别识别不仅在商业领域（如精准营销、客户服务）具有重要应用价值，还在社会安全、身份认证等方面发挥着关键作用。通过本实训，学员将学习到如何从数据收集、预处理到模型构建、训练与评估的完整流程，掌握深度学习在人脸识别领域的应用方法，为未来在人工智能相关领域的研究和开发奠定坚实基础。

## 1. 项目背景介绍


数据集来源：通过网络收集全身图片，再通过 opencv 提供的人脸检测模型，截取人脸部分作为数据集  
数据集位置：/home/jovyan/work/datasets/689457a87fedd2132f427a9e-momodel

其中 data 表示已经处理过后的图像
ori_data 表示收集到的源图像


项目文件说明

*    `data_clipping.py` ：检测人脸并截取人脸区域图像
*    `haarcascade_frontalface_default.xml` ：开源的人脸检测模型
*    `model.py`　：　模型定义文件
*    `main.py` : 模型训练代码
*    `detection.py` ： 项目演示用来检测训练的模型的预测效果




In [ ]:
!pip install torchsummary


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import model
import dataset
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm
import random

seed = 1234
random.seed(seed)
np.random.seed(seed)



## 2. 数据集收集与分析

开始项目前，我们先分析数据集的构成，已经将不同性别的人脸放在不同文件夹下并命名为 0/1，下载的原始图片已保存在 `/home/jovyan/work/datasets/689457a87fedd2132f427a9e-momodel` 目录下

In [ ]:
def show_5_image():
    """
    显示数据集中的5个数据,这里是经过数据预处理后的图像数据
    """
    data_path = '/home/jovyan/work/datasets/689457a87fedd2132f427a9e-momodel/data/train_data'
    data = dataset.datasets(data_path)
    plt.figure()
    for i, (data, label) in enumerate(data):
        if i >= 5:
            break
        plt.subplot(2, 3, i + 1)
        plt.title("label:" + str(label))
        show_image = np.transpose(np.array(data, np.float32), (1, 2, 0))
        plt.imshow(show_image)
        plt.axis("off")
#         plt.savefig("../../results/data_sample.png")
    plt.show()

show_5_image()


## 3. 数据集构建

In [ ]:
class datasets(Dataset):
    def __init__(self, data_path='', transform=None, train=True):
        super(datasets, self).__init__()
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize(224, ),
                transforms.ToTensor(),
                transforms.Normalize(0.5, 0.5), ])
        else:
            self.transform = transform
        class_path = os.listdir(data_path)
        self.data = []
        self.labels = []
        for idx, classes in enumerate(class_path):
            images = os.listdir(data_path + '/' + classes)
            for image in images:
                self.data.append(data_path + '/' + classes + '/' + image)
                self.labels.append(idx)

        # 打乱数据
        state = np.random.get_state()
        np.random.shuffle(self.data)
        np.random.set_state(state)
        np.random.shuffle(self.labels)

        # 划分训练验证集
        if train:
            self.data = self.data[:int(len(self.data)*0.7)]
            self.labels = self.labels[:int(len(self.labels)*0.7)]
        else:
            self.data = self.data[int(len(self.data) * 0.7):]
            self.labels = self.labels[int(len(self.labels) * 0.7):]

    def __getitem__(self, item):
        image = Image.open(self.data[item])
        # print(image)
        image_mat = self.transform(image)
        return image_mat, self.labels[item]

    def __len__(self):
        return len(self.data)



## 4. 模型构建
本项目中模型采用 resnet 模型，将模型可视化，参数大小 42 Mb



In [ ]:
import torch
from torchvision import models
import torch.nn as nn
import torchsummary


def Resnet101():
    model = model = models.resnet101()
    model.fc = nn.Sequential(nn.Linear(2048, 1000),
                             nn.ReLU(),
                             nn.Linear(1000, 512),
                             nn.ReLU(),
                             nn.Linear(512, 2))
    return model


def Resnet50():
    model = model = models.resnet50()
    model.fc = nn.Sequential(nn.Linear(2048, 1000),
                             nn.ReLU(),
                             nn.Linear(1000, 512),
                             nn.ReLU(),
                             nn.Linear(512, 2))
    return model


def Resnet34():
    model = model = models.resnet34()
    model.fc = nn.Sequential(nn.Linear(512, 2))
    return model


def Resnet18():
    model = model = models.resnet18()
    model.fc = nn.Sequential(nn.Linear(512, 2))
    return model


x = torch.randn(4, 3, 224, 224)
model = Resnet50()
print(model(x).shape)
torchsummary.summary(model, input_size=(3, 224, 224), batch_size=4, device='cpu')



## 5. 模型的训练与测试


- 使用交叉熵作为损失函数
- 使用 Adam 作为优化器

In [ ]:
data_path = '/home/jovyan/work/datasets/689457a87fedd2132f427a9e-momodel/data/train_data'
max_epoch = 30
batch_size = 32
lr = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_data = dataset.datasets(data_path=data_path, train=True)
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

val_data = dataset.datasets(data_path=data_path, train=False)
val_dataloader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_func = nn.CrossEntropyLoss()


In [ ]:
def train(name):
    train_loss = []
    val_loss = []
    val_acc = []
    min_val_acc = np.inf
    best_model = None
    for epoch in range(max_epoch):
        model.train()
        epoch_train_loss = []
        epoch_val_loss = []
        epoch_val_acc = []

        t = tqdm(train_dataloader)
        for x, y in t:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            y_ = model(x)
            loss = loss_func(y_, y)
            loss.backward()
            optimizer.step()
            t.set_postfix(train_loss=loss.item())
            epoch_train_loss.append(loss.item())

        model.eval()
        t = tqdm(val_dataloader)
        for x, y in t:
            x, y = x.to(device), y.to(device)
            y_ = model(x)
            loss = loss_func(y_, y)
            pred = torch.max(y_, dim=1)[1]
            acc = float(torch.eq(y.cpu(), pred.cpu()).sum() / len(x))
            t.set_postfix(val_loss=loss.item(), val_acc=acc)
            epoch_val_loss.append(loss.item())
            epoch_val_acc.append(acc)

        train_loss.append(np.mean(epoch_train_loss))
        val_loss.append(np.mean(epoch_val_loss))
        val_acc.append(np.mean(epoch_val_acc))
        print(f'{epoch + 1}|{max_epoch} train_loss={train_loss[-1]}')
        print(f'{epoch + 1}|{max_epoch} val_loss={val_loss[-1]}')
        print(f'{epoch + 1}|{max_epoch} val_acc={val_acc[-1]}')

        # 保存最好的模型
        if val_loss[-1] < min_val_acc:
            min_val_acc = val_acc[-1]
            best_model = model
            torch.save(best_model, f'/home/jovyan/work/results/best_{name}.pt')

    torch.save(model, f'/home/jovyan/work/results/last_{name}.pt')

    plt.figure()
    plt.plot(train_loss, label='train_loss')
    plt.plot(val_loss, label='val_loss')
    plt.plot(val_acc, label='val_acc')
    plt.legend()
    plt.show()


使用 gpu 进行模型训练，代码在 `train.py` 文件中  
使用 Resnet50 模型训练 50 个迭代次数后，模型在验证集上准确率达到94.57%，且测试损失下降至0.21

## 6. 模型效果测试

In [ ]:
test_data = dataset.datasets(data_path='/home/jovyan/work/datasets/689457a87fedd2132f427a9e-momodel/data/test_data', train=True)
test_dataloader = DataLoader(test_data, batch_size=4, shuffle=False)


In [ ]:
model = torch.load('/home/jovyan/work/results/best_resnet50.pt', map_location=torch.device('cpu'))

device = 'cpu'
epoch_val_acc = []
model.eval()
t = tqdm(test_dataloader)
for x, y in t:
    x, y = x.to(device), y.to(device)
    y_ = model(x)
    pred = torch.max(y_, dim=1)[1]
    acc = float(torch.eq(y.cpu(), pred.cpu()).sum() / len(x))
    t.set_postfix(val_acc=acc)
    epoch_val_acc.append(acc)

print('val_acc', np.mean(epoch_val_acc))


模型在测试集上准确率为 0.72


## 7. 检测并演示效果

选用真人照片进行测试，代码如下

In [ ]:
import matplotlib.pyplot as plt
import torch
import cv2
import torch.nn.functional as f
from PIL import Image
from torchvision import transforms
plt.rcParams['font.family'] = ['SimHei']


def detection_one(test_image, model_path, classifier='haarcascade_frontalface_default.xml', transform=None):
    face_cascade = cv2.CascadeClassifier(classifier)
    if test_image[-3:] == 'jpg':
        ori_image = cv2.imread(test_image)
        ori_image = cv2.cvtColor(ori_image, cv2.COLOR_BGR2RGB)
        faces = face_cascade.detectMultiScale(ori_image, 1.3, 5)
        if len(faces) == 0:
            print('未检测到人脸')
            return
        elif len(faces) == 1:
            # 画出检测结果
            detection_image = None
            for (x, y, w, h) in faces:
                ori_image = cv2.rectangle(ori_image, (x, y), (x + w, y + h), (255, 0, 0), 2)
                detection_image = ori_image[y + 2:y + h, x + 2:x + w]

            # 选出检测区域进行识别
            show_image = ori_image
            detection_image = Image.fromarray(detection_image)

            model = torch.load(model_path, map_location=torch.device('cpu'))
            model.eval()
            image = detection_image

            if transform is None:
                transform = transforms.Compose([
                    transforms.Resize(224, ),
                    transforms.ToTensor(),
                    transforms.Normalize(0.5, 0.5), ])
            else:
                transform = transform

            classes = ['女', '男']
            image = transform(image)
            image = image.unsqueeze(0)
            y_ = model(image)
            pred = torch.max(y_, dim=1)[1]
            # print(image.shape)
            print(classes[int(pred)])

            plt.figure()
            plt.title(f'预测结果:{classes[int(pred)]}')
            plt.imshow(show_image)
            plt.axis('off')
            plt.show()
        else:
            print('检测到多个人脸')
            return
    elif test_image[:6] == 'camera':
        capture = cv2.VideoCapture(int(test_image[-1]))
        model = torch.load(model_path, map_location=torch.device('cpu'))
        model.eval()
        while True:
            c = cv2.waitKey(1)
            if c == 27:
                break
            res, frame = capture.read()
            ori_image = frame
            show_image = ori_image
            faces = face_cascade.detectMultiScale(ori_image, 1.3, 5)

            if len(faces) == 0:
                print('未检测到人脸')
            elif len(faces) == 1:
                # 画出检测结果
                detection_image = None
                for (x, y, w, h) in faces:
                    ori_image = cv2.rectangle(ori_image, (x, y), (x + w, y + h), (255, 0, 0), 2)
                    detection_image = ori_image[y + 2:y + h, x + 2:x + w]

                # 选出检测区域进行识别
                show_image = ori_image
                detection_image = Image.fromarray(detection_image)
                image = detection_image

                if transform is None:
                    transform = transforms.Compose([
                        transforms.Resize(224, ),
                        transforms.ToTensor(),
                        transforms.Normalize(0.5, 0.5), ])
                else:
                    transform = transform

                classes = ['woman', 'man']
                image = transform(image)
                image = image.unsqueeze(0)
                y_ = model(image)
                Soft = f.softmax(y_).detach().numpy()[0]
                pred = torch.max(y_, dim=1)[1]

                # print(image.shape)
                font = cv2.FONT_HERSHEY_SIMPLEX
                show_image = cv2.putText(show_image, classes[int(pred)], (100, 100), font, 2, (0, 255, 0), 3)
                print('置信度', Soft[int(pred)])
                print(classes[int(pred)])
            else:
                print('检测到多个人脸')
            cv2.imshow('show_result', show_image)
        capture.release()
        cv2.destroyAllWindows()

detection_one('/home/jovyan/work/kun.jpg', '/home/jovyan/work/results/best_resnet50.pt')


可见模型能够根据图片中的人脸正确预测其性别。

## 8. 分析与总结
本文提到的人脸性别识别算法使用 pytorch 框架实现，相对简单，但面对一些临近边界的样本会产生一定的偏差，因此改进的方式如下：


1.   数据集上增强样本的多样性，防止其只在标准的图片上进行区分，使得模型更鲁棒
2.   在网络模型结构上进行改进，加入注意力机制，抓取全局的特征辅助识别
3.   同样是在网络模型上进行改进，也可以使用生成模型去进行人脸性别识别，例如 GAN，扩散模型等

